In [ ]:
import networkx as nx
import random
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

# ---------------------------
#  Load graph
# ---------------------------
G = nx.read_graphml("hetionet.graphml")
G_u = G

## Link prediction binary

#RGCN


#link Prediction Binary

In [ ]:


# ===============================
# 0) Imports & device
# ===============================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from collections import defaultdict
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===============================
# 1) Graph preprocessing (G assumed to be loaded)
# ===============================
# Node mappings
node_type_map = defaultdict(list)
node_id_map = {}
node_idx_by_type = defaultdict(dict)
for nid, data in G.nodes(data=True):
    ntype = data.get("kind", "Unknown")
    idx = len(node_type_map[ntype])
    node_type_map[ntype].append(nid)
    node_id_map[nid] = (ntype, idx)
    node_idx_by_type[ntype][nid] = idx

# Edge mappings
edge_type_map = defaultdict(list)
for u, v, data in G.edges(data=True):
    rel = data.get("metaedge", data.get("relation", None))
    if not rel or u not in node_id_map or v not in node_id_map:
        continue
    u_t, u_i = node_id_map[u]
    v_t, v_i = node_id_map[v]
    edge_type_map[(u_t, rel, v_t)].append((u_i, v_i))

print(f"Node types: {list(node_type_map.keys())}")
print(f"Edge types: {len(edge_type_map)}")

# ===============================
# 2) Build homogeneous graph tensors
# ===============================
type_offsets = {}
global_id_map = {}
offset = 0
for ntype, nodes in node_type_map.items():
    type_offsets[ntype] = offset
    for i in range(len(nodes)):
        global_id_map[(ntype, i)] = offset + i
    offset += len(nodes)
num_nodes = offset

edge_type_keys = list(edge_type_map.keys())
rel2id = {etype: i for i, etype in enumerate(edge_type_keys)}
num_relations = len(edge_type_keys)

all_src, all_dst, all_rel = [], [], []
for etype, edges in edge_type_map.items():
    rel_id = rel2id[etype]
    src_type, _, dst_type = etype
    for s_local, d_local in edges:
        all_src.append(type_offsets[src_type] + s_local)
        all_dst.append(type_offsets[dst_type] + d_local)
        all_rel.append(rel_id)

edge_index = torch.tensor([all_src, all_dst], dtype=torch.long, device=device)
edge_type  = torch.tensor(all_rel, dtype=torch.long, device=device)
print(f"Graph tensor -> nodes: {num_nodes}, edges: {edge_index.size(1)}")

# ===============================
# 3) Per-relation train/val/test splits
# ===============================
edge_type_splits = {}
for etype in edge_type_keys:
    rel_id = rel2id[etype]
    mask = (edge_type == rel_id)
    edges = edge_index[:, mask]
    n_edges = edges.size(1)
    if n_edges < 50:  # skip too small
        continue
    perm = torch.randperm(n_edges)
    n_train = int(0.8 * n_edges)
    n_val = int(0.1 * n_edges)
    train_idx = perm[:n_train]
    val_idx = perm[n_train:n_train+n_val]
    test_idx = perm[n_train+n_val:]
    edge_type_splits[etype] = {
        "train": edges[:, train_idx],
        "val": edges[:, val_idx],
        "test": edges[:, test_idx],
        "all": edges
    }

# ===============================
# 4) RGCN model
# ===============================
class RGCNNet(nn.Module):
    def __init__(self, num_nodes, out_dim=64, num_relations=num_relations):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, 64)
        self.rgcn1 = RGCNConv(64, 64, num_relations, num_bases=None)
        self.rgcn2 = RGCNConv(64, out_dim, num_relations, num_bases=None)

    def forward(self, node_ids, edge_index, edge_type):
        x = self.emb(node_ids)
        x = F.relu(self.rgcn1(x, edge_index, edge_type))
        x = self.rgcn2(x, edge_index, edge_type)
        return x

def edge_scores(z, eidx):
    return torch.sigmoid((z[eidx[0]] * z[eidx[1]]).sum(dim=-1))

# ===============================
# 5) Negative sampling
# ===============================
def build_pools_and_forbidden(edge_type_splits, node_type_map, type_offsets):
    rel_resources = {}
    for rel_key, splits in edge_type_splits.items():
        src_t, _, dst_t = rel_key
        src_pool = list(range(type_offsets[src_t], type_offsets[src_t] + len(node_type_map[src_t])))
        dst_pool = list(range(type_offsets[dst_t], type_offsets[dst_t] + len(node_type_map[dst_t])))
        forbidden = set((int(s), int(d)) for s, d in zip(splits["all"][0].tolist(), splits["all"][1].tolist()))
        rel_resources[rel_key] = {"src_pool": src_pool, "dst_pool": dst_pool, "forbidden": forbidden}
    return rel_resources

def sample_type_consistent_negs(num_samples, src_pool, dst_pool, forbidden_set, device):
    src_pool_t = torch.tensor(src_pool, device=device)
    dst_pool_t = torch.tensor(dst_pool, device=device)
    neg_src, neg_dst = [], []
    while len(neg_src) < num_samples:
        s = src_pool_t[torch.randint(0, len(src_pool_t), (num_samples*2,))]
        d = dst_pool_t[torch.randint(0, len(dst_pool_t), (num_samples*2,))]
        pairs = [(int(si), int(di)) for si, di in zip(s, d)]
        filtered = [p for p in pairs if p not in forbidden_set]
        if len(filtered) > 0:
            ns = [p[0] for p in filtered][:num_samples - len(neg_src)]
            nd = [p[1] for p in filtered][:num_samples - len(neg_dst)]
            neg_src.extend(ns)
            neg_dst.extend(nd)
    return torch.tensor([neg_src, neg_dst], device=device)

rel_resources = build_pools_and_forbidden(edge_type_splits, node_type_map, type_offsets)

# ===============================
# 6) Train unified RGCN
# ===============================
node_ids = torch.arange(num_nodes, device=device)
model = RGCNNet(num_nodes, out_dim=64, num_relations=num_relations).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
EPOCHS = 20

for epoch in range(1, EPOCHS+1):
    model.train()
    optimizer.zero_grad()
    z = model(node_ids, edge_index, edge_type)
    pos_edges_all, neg_edges_all = [], []
    for rel_key, splits in edge_type_splits.items():
        pos_e = splits["train"]
        if pos_e.size(1) == 0:
            continue
        N = pos_e.size(1)
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        pos_edges_all.append(pos_e)
        neg_edges_all.append(neg_e)
    pos_edges_all = torch.cat(pos_edges_all, dim=1)
    neg_edges_all = torch.cat(neg_edges_all, dim=1)
    pos_scores = edge_scores(z, pos_edges_all)
    neg_scores = edge_scores(z, neg_edges_all)
    loss = -(pos_scores+1e-15).log().mean() - (1-neg_scores+1e-15).log().mean()
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# ===============================
# 7) Evaluation per relation
# ===============================
model.eval()
results = []
with torch.no_grad():
    z = model(node_ids, edge_index, edge_type)
    for rel_key, splits in edge_type_splits.items():
        N = splits["test"].size(1)
        if N == 0: continue
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        y_true = torch.cat([torch.ones(N, device=device), torch.zeros(N, device=device)])
        scores = torch.cat([edge_scores(z, splits["test"]), edge_scores(z, neg_e)])
        y = y_true.cpu().numpy()
        s = scores.cpu().numpy()
        auc = roc_auc_score(y, s)
        thr = 0.5
        preds = (s > thr).astype(int)
        p = precision_score(y, preds, zero_division=0)
        r = recall_score(y, preds, zero_division=0)
        f1 = f1_score(y, preds, zero_division=0)
        results.append((rel_key, auc, p, r, f1, thr))

print("\n📊 Final evaluation per edge type:")
for rel_key, auc, p, r, f1, thr in results:
    print(f"{rel_key}: AUC={auc:.4f}, P={p:.3f}, R={r:.3f}, F1={f1:.3f}, Thr={thr:.2f}")

# ===============================
# 8) Overall evaluation
# ===============================
all_pos, all_neg = [], []
with torch.no_grad():
    for rel_key, splits in edge_type_splits.items():
        N = splits["test"].size(1)
        if N == 0: continue
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        all_pos.append(splits["test"])
        all_neg.append(neg_e)

all_pos = torch.cat(all_pos, dim=1)
all_neg = torch.cat(all_neg, dim=1)

with torch.no_grad():
    z = model(node_ids, edge_index, edge_type)
    scores_pos = edge_scores(z, all_pos)
    scores_neg = edge_scores(z, all_neg)

y_true = torch.cat([torch.ones(scores_pos.numel(), device=device),
                    torch.zeros(scores_neg.numel(), device=device)])
scores_all = torch.cat([scores_pos, scores_neg], dim=0)
thr_overall = 0.5
preds_overall = (scores_all > thr_overall).int()

auc_overall = roc_auc_score(y_true.cpu().numpy(), scores_all.cpu().numpy())
p_overall   = precision_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)
r_overall   = recall_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)
f1_overall  = f1_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)

print("\n🧮 Overall performance across all relations:")
print(f"AUC = {auc_overall:.4f}, Precision = {p_overall:.3f}, Recall = {r_overall:.3f}, F1 = {f1_overall:.3f}")


Using device: cuda
Node types: ['Anatomy', 'Biological Process', 'Cellular Component', 'Compound', 'Disease', 'Gene', 'Molecular Function', 'Pathway', 'Pharmacologic Class', 'Side Effect', 'Symptom', 'Unknown']
Edge types: 20
Graph tensor -> nodes: 47033, edges: 1379933
Epoch 1 | Loss: 27.4898
Epoch 5 | Loss: 21.2778
Epoch 10 | Loss: 11.6912
Epoch 15 | Loss: 6.0136
Epoch 20 | Loss: 3.5757

📊 Final evaluation per edge type:
('Anatomy', 'AdG', 'Gene'): AUC=0.8298, P=0.708, R=0.886, F1=0.787, Thr=0.50
('Anatomy', 'AeG', 'Gene'): AUC=0.8186, P=0.706, R=0.834, F1=0.765, Thr=0.50
('Anatomy', 'AuG', 'Gene'): AUC=0.8340, P=0.705, R=0.895, F1=0.789, Thr=0.50
('Compound', 'CrC', 'Compound'): AUC=0.7274, P=0.541, R=0.926, F1=0.683, Thr=0.50
('Compound', 'CtD', 'Disease'): AUC=0.6477, P=0.513, R=0.789, F1=0.622, Thr=0.50
('Compound', 'CbG', 'Gene'): AUC=0.6792, P=0.603, R=0.715, F1=0.655, Thr=0.50
('Compound', 'CuG', 'Gene'): AUC=0.6748, P=0.588, R=0.711, F1=0.644, Thr=0.50
('Compound', 'CdG', 'Ge